In [ ]:
import pygame
import sys
import random
import numpy as np
import os

pygame.init()  # Pygame başlatılıyor

# ---------------------------------------
# GRID AYARLARI
# ---------------------------------------
GRID = 6             # Grid boyutu (6x6)
CELL = 100           # Her hücrenin piksel boyutu
SCREEN_W = GRID * CELL + 600  # Ekran genişliği (Q-table alanı için ekstra)
SCREEN_H = GRID * CELL        # Ekran yüksekliği
screen = pygame.display.set_mode((SCREEN_W, SCREEN_H))  # Pencere oluştur
pygame.display.set_caption("Taxi Q-Learning Simulation")
clock = pygame.time.Clock()  # FPS kontrolü için clock

# ---------------------------------------
# DURAKLAR
# ---------------------------------------
stops = [
    (0, 0, "A"),  # Üst sol köşe
    (0, 5, "B"),  # Üst sağ köşe
    (5, 0, "C"),  # Alt sol köşe
    (5, 5, "D")   # Alt sağ köşe
]

# ---------------------------------------
# ENGELLER
# ---------------------------------------
walls = [
    (1, 1, "RIGHT"),  # (satır, sütun, yön)
    (2, 2, "DOWN"),
    (3, 3, "UP"),
    (4, 1, "LEFT"),
]

# ---------------------------------------
# Q-LEARNING PARAMETRELER
# ---------------------------------------
alpha = 0.1                 # Öğrenme hızı
gamma = 0.95                # Gelecek ödül indirgeme faktörü
epsilon = 0.1               # Keşfetme olasılığı
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]  # Yukarı, aşağı, sol, sağ
ACTION_COUNT = 4
Q = np.zeros((GRID, GRID, 4, 4, ACTION_COUNT))  # Q-table: r,c,passenger,destination,action
if os.path.exists("qtable.npy"):
    Q = np.load("qtable.npy")  # Q-table varsa yükle
episode_count = 0            # Episode sayacı
episode_finished = True      # Başlangıçta episode bitmiş sayılır

# ---------------------------------------
# ENGEL KONTROL FONKSİYONLARI
# ---------------------------------------
def wall_exists(r,c,direction):
    return (r,c,direction) in walls  # Belirtilen hücrede duvar var mı?

def can_move(r,c,action):
    dr,dc = ACTIONS[action]
    nr,nc = r+dr,c+dc
    if nr<0 or nr>=GRID or nc<0 or nc>=GRID:  # Grid dışına çıkma kontrolü
        return False
    # Hareket yönünde duvar kontrolü
    if action==0 and wall_exists(r,c,"UP"): return False
    if action==1 and wall_exists(r,c,"DOWN"): return False
    if action==2 and wall_exists(r,c,"LEFT"): return False
    if action==3 and wall_exists(r,c,"RIGHT"): return False
    # Hedef hücrede ters yönde duvar kontrolü
    if action==0 and wall_exists(nr,nc,"DOWN"): return False
    if action==1 and wall_exists(nr,nc,"UP"): return False
    if action==2 and wall_exists(nr,nc,"RIGHT"): return False
    if action==3 and wall_exists(nr,nc,"LEFT"): return False
    return True

# ---------------------------------------
# GRAFİK FONKSİYONLARI
# ---------------------------------------
def draw_grid():
    for r in range(GRID):
        for c in range(GRID):
            rect = pygame.Rect(c*CELL,r*CELL,CELL,CELL)  # Hücre dikdörtgeni
            pygame.draw.rect(screen,(120,120,120),rect)   # Hücre dolgu rengi
            pygame.draw.rect(screen,(180,180,180),rect,2) # Hücre sınırı

def draw_walls():
    thick=10
    color=(30,30,30)
    for r,c,d in walls:
        x=c*CELL
        y=r*CELL
        # Duvar çizimleri
        if d=="UP": pygame.draw.line(screen,color,(x,y),(x+CELL,y),thick)
        if d=="DOWN": pygame.draw.line(screen,color,(x,y+CELL),(x+CELL,y+CELL),thick)
        if d=="LEFT": pygame.draw.line(screen,color,(x,y),(x,y+CELL),thick)
        if d=="RIGHT": pygame.draw.line(screen,color,(x+CELL,y),(x+CELL,y+CELL),thick)

def draw_stops():
    font = pygame.font.SysFont("Arial",40,bold=True)
    colors = [(200,80,80),(80,200,80),(80,80,200),(200,200,80)]
    for i,(r,c,label) in enumerate(stops):
        rect=pygame.Rect(c*CELL+15,r*CELL+15,CELL-30,CELL-30)
        pygame.draw.rect(screen,colors[i],rect,border_radius=15)  # Durak kutusu
        text=font.render(label,True,(0,0,0))
        screen.blit(text,text.get_rect(center=rect.center))       # Durak etiketi

def draw_taxi(r,c):
    rect=pygame.Rect(c*CELL+25,r*CELL+25,CELL-50,CELL-50)
    pygame.draw.rect(screen,(255,255,255),rect,border_radius=10)  # Taxi

def draw_passenger(r,c):
    center=(c*CELL+CELL//2,r*CELL+CELL//2)
    pygame.draw.circle(screen,(0,0,0),center,22)        # Yolcu ana daire
    pygame.draw.circle(screen,(230,230,230),center,18)  # İç daire
    pygame.draw.circle(screen,(0,0,0),(center[0],center[1]-8),5)  # Baş
    pygame.draw.line(screen,(0,0,0),(center[0],center[1]-3),(center[0],center[1]+10),3) # Gövde
    pygame.draw.line(screen,(0,0,0),(center[0],center[1]+10),(center[0]-7,center[1]+20),3) # Sol bacak
    pygame.draw.line(screen,(0,0,0),(center[0],center[1]+10),(center[0]+7,center[1]+20),3) # Sağ bacak

def draw_passenger_goal(dest_idx):
    r,c,_=stops[dest_idx]
    rect=pygame.Rect(c*CELL+40,r*CELL+40,20,20)
    pygame.draw.rect(screen,(255,0,0),rect)  # Hedef durağı kırmızı kare ile göster

# ---------------------------------------
# Q-TABLE KENAR OKLARI + SAYISAL (Çakışma yok)
# ---------------------------------------
def draw_qtable_grid_edges(pass_idx,dest_idx):
    font = pygame.font.SysFont("Arial", 20)
    x0 = GRID * CELL + 20  # Başlangıç x
    y0 = 20                # Başlangıç y
    cell_size = 80          # Hücre boyutu, sayılar ve oklar için geniş

    for r in range(GRID):
        for c in range(GRID):
            qvals = Q[r, c, pass_idx, dest_idx]
            qvals_rounded = np.round(qvals, 1)
            cell_x = x0 + c*(cell_size + 15)
            cell_y = y0 + r*(cell_size + 15)

            # Yukarı ok ve değer
            screen.blit(font.render("▲"+str(qvals_rounded[0]), True, (255,255,255)),
                        (cell_x + cell_size//2 - 10, cell_y))

            # Aşağı ok ve değer
            screen.blit(font.render("▼"+str(qvals_rounded[1]), True, (255,255,255)),
                        (cell_x + cell_size//2 - 10, cell_y + cell_size - 25))

            # Sol ok ve değer
            screen.blit(font.render("◄"+str(qvals_rounded[2]), True, (255,255,255)),
                        (cell_x, cell_y + cell_size//2 - 10))

            # Sağ ok ve değer
            screen.blit(font.render("►"+str(qvals_rounded[3]), True, (255,255,255)),
                        (cell_x + cell_size - 35, cell_y + cell_size//2 - 10))

# ---------------------------------------
# Q-LEARNING FONKSİYONLARI
# ---------------------------------------
def choose_action(r,c,p,d):
    if random.random()<epsilon:                     # Epsilon-greedy
        return random.randint(0,ACTION_COUNT-1)    # Keşfet
    return np.argmax(Q[r,c,p,d])                   # Maksimal Q değeri

def reset_episode():
    taxi_r=random.randint(0,GRID-1)
    taxi_c=random.randint(0,GRID-1)
    passenger_idx=random.randint(0,3)
    destination_idx=random.randint(0,3)
    while destination_idx==passenger_idx:
        destination_idx=random.randint(0,3)
    onboard=False
    return taxi_r,taxi_c,passenger_idx,destination_idx,onboard

# ---------------------------------------
# ANA DÖNGÜ
# ---------------------------------------
taxi_r=taxi_c=0
pass_idx=dest_idx=0
onboard=False

while True:
    # Eventler (çıkış ve SPACE)
    for event in pygame.event.get():
        if event.type==pygame.QUIT:
            np.save("qtable.npy",Q)   # Q-table kaydet
            pygame.quit()
            sys.exit()
        if event.type==pygame.KEYDOWN:
            if event.key==pygame.K_SPACE and episode_finished:
                taxi_r,taxi_c,pass_idx,dest_idx,onboard=reset_episode()
                episode_finished=False
                episode_count+=1

    # Çizim
    screen.fill((50,50,50))
    draw_grid()
    draw_walls()
    draw_stops()
    draw_taxi(taxi_r,taxi_c)
    draw_passenger_goal(dest_idx)

    if not onboard:
        pr,pc,_=stops[pass_idx]
        draw_passenger(pr,pc)

    draw_qtable_grid_edges(pass_idx,dest_idx)  # Yeni Q-table çizimi

    if episode_finished:
        font=pygame.font.SysFont("Arial",60,bold=True)
        text=font.render("PRESS SPACE",True,(255,255,255))
        screen.blit(text,(80,SCREEN_H//2-30))
        pygame.display.flip()
        clock.tick(60)
        continue

    # RL hareketi ve Q-learning update
    old_r,old_c=taxi_r,taxi_c
    action=choose_action(taxi_r,taxi_c,pass_idx,dest_idx)

    if can_move(taxi_r,taxi_c,action):
        dr,dc=ACTIONS[action]
        taxi_r+=dr
        taxi_c+=dc

    reward=-1
    if not onboard:  # Yolcu alımı
        pr,pc,_=stops[pass_idx]
        if taxi_r==pr and taxi_c==pc:
            onboard=True
            reward=20
    else:           # Yolcuyu bırakma
        dr,dc,_=stops[dest_idx]
        if taxi_r==dr and taxi_c==dc:
            reward=50
            episode_finished=True

    best_next=np.max(Q[taxi_r,taxi_c,pass_idx,dest_idx])
    Q[old_r,old_c,pass_idx,dest_idx,action]+=alpha*(reward+gamma*best_next-Q[old_r,old_c,pass_idx,dest_idx,action])

    pygame.display.flip()
    clock.tick(5)

    if episode_finished:
        np.save("qtable.npy",Q)


C:\Users\Public\anaconda3\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
